# UAE Rental Price Prediction — 03 Feature Engineering

**Input:** `UAE_properties_clean_python.csv` (Notebook 01) or `UAE_properties_clean.csv` (RapidMiner).  
**Output:** `UAE_properties_features.csv` for Notebooks 04, 05 and 06.

We create a few **property-based** features using size, bedrooms, and bathrooms. We do **not** create features from rental price, because that would leak the answer into the classification model.

**Run the cells from top to bottom.** Calculated results appear only after you run the notebook using your dataset.

## 1. Load the cleaned data

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

# Search the same folders in GitHub, Jupyter, and Google Colab.
paths = [
    Path('../data/UAE_properties_clean_python.csv'),
    Path('../data/UAE_properties_clean.csv'),
    Path('data/UAE_properties_clean_python.csv'),
    Path('data/UAE_properties_clean.csv'),
    Path('UAE_properties_clean_python.csv'),
    Path('UAE_properties_clean.csv'),
    Path('/content/UAE_properties_clean_python.csv'),
    Path('/content/UAE_properties_clean.csv'),
]
source_path = next((p for p in paths if p.is_file()), None)
if source_path is None:
    raise FileNotFoundError('Cleaned CSV not found. Run Notebook 01 and upload its CSV, or upload UAE_properties_clean.csv.')
df = pd.read_csv(source_path, low_memory=False)
print('Loaded:', source_path)
print('Rows, columns:', df.shape)
print('Missing cells:', int(df.isna().sum().sum()))
assert 'Rent' in df and 'Rent_category' in df, 'Both targets must be preserved in the shared dataset.'
display(df.head(3))

## 2. Check the original modeling inputs

The shared dataset contains the two targets plus numerical and dummy-coded property information.

In [ ]:
print('Columns:')
for c in df.columns:
    print('-',c)
print('Rent categories:')
display(df['Rent_category'].value_counts(dropna=False).to_frame('Count'))
print('Missing data:',int(df.isna().sum().sum()))

## 3. Create useful features

- `Sqft_per_bedroom`: floor area divided by the number of bedrooms (for a studio, use 1).
- `Bathrooms_per_bedroom`: bathroom-to-bedroom ratio (for a studio, use 1).
- `Total_rooms`: bedrooms plus bathrooms.
- `Is_studio`: 1 when the listing has zero bedrooms.

No new feature uses `Rent` or `Rent_category`.

In [ ]:
features_df = df.copy()
for c in ['Beds','Baths','Area_in_sqft','Age_of_listing_in_days']:
    features_df[c] = pd.to_numeric(features_df[c], errors='coerce')

# Divide by at least 1 to avoid division by zero for studio apartments.
bedroom_denominator = features_df['Beds'].clip(lower=1)
features_df['Sqft_per_bedroom'] = features_df['Area_in_sqft'] / bedroom_denominator
features_df['Bathrooms_per_bedroom'] = features_df['Baths'] / bedroom_denominator
features_df['Total_rooms'] = features_df['Beds'] + features_df['Baths']
features_df['Is_studio'] = (features_df['Beds'] == 0).astype(int)

new_cols = ['Sqft_per_bedroom','Bathrooms_per_bedroom','Total_rooms','Is_studio']
print('New features:',new_cols)
display(features_df[['Beds','Baths','Area_in_sqft'] + new_cols].head(8))

## 4. Validate the final feature table

In [ ]:
assert 'Rent' in features_df and 'Rent_category' in features_df
assert not features_df.columns.duplicated().any()
assert features_df['Rent'].gt(0).all()
assert features_df[new_cols].notna().all().all()
assert np.isfinite(features_df[new_cols].to_numpy(dtype=float)).all()
print('Original rows:', len(df), '| Rows after feature engineering:',len(features_df))
print('Original columns:', df.shape[1], '| Columns after feature engineering:',features_df.shape[1])
print('Number of new features:',len(new_cols))
print('New columns do not depend on Rent or Rent_category.')

## 5. Save the feature-engineered CSV

Run this notebook before training models. The CSV is written next to the cleaned input file.

In [ ]:
output_path = source_path.parent / 'UAE_properties_features.csv'
features_df.to_csv(output_path,index=False)
print('Saved:',output_path)
print('File size (MB):',round(output_path.stat().st_size / 1024**2,2))
print('Ready for Notebooks 04, 05 and 06.')

## Summary for the team

- Feature engineering added four explanatory property features.
- We kept both targets in the shared file, but **each modeling notebook excludes its own leakage columns**.
- Later notebooks use training/testing or clustering-specific processing.
- We will discuss whether these added features actually improve the models after evaluation.